# gold_base

Base comum da camada **Gold**: monta uma vez as dimensões e o fato central que as outras Gold usam, para cada número ter uma única definição. Na Gold, ele faz o papel que o `silver_cadastro` faz na Silver: roda primeiro e, se falhar, nenhuma Gold roda.

- **Entrada:** `silver.tb_unidades_consumidoras` e `silver.tb_medidores` (silver_cadastro), `silver.tb_faturas` (silver_faturamento), `silver.tb_leituras` (silver_medicao) e `silver.tb_geracao_distribuida` (silver_perdas).
- **Saída:** `gold.dim_mes`, `gold.dim_uc`, `gold.dim_alimentador` e `gold.fato_uc_mes`.
- **Alimenta:** `gold_receita`, `gold_kpis`, `gold_dec_fec`, `gold_fila_perdas` e `gold_visao_360`.
- Segue o documento *Decisões da Engenharia de Dados: camada Gold* (v2, seções 3.1 a 3.3). Idempotente: grava em overwrite e pode ser reexecutado.

| Tabela | Grão (1 linha por) | Conteúdo |
|---|---|---|
| `dim_mes` | mês (gerado de `sequence` + `explode`) | `mes_referencia`, `ano`, `mes`, `trimestre`, `periodo` (P1/P2) e `bandeira_tarifaria` do mês |
| `dim_uc` | UC | cadastro + medidor ativo (`id_medidor`, `tipo_medidor`, `modelo_medidor`, `data_instalacao_medidor`) + GD (`gd_*`) + `flag_tem_gd`, `flag_trocou_medidor` e as flags da Silver |
| `dim_alimentador` | alimentador | `municipio`, `ucs_atendidas` (o `Cc` do DEC/FEC) e `bairros_atendidos` |
| `fato_uc_mes` | UC × mês | todas as colunas da fatura e da leitura da mesma UC e mês, com as flags das duas |

**Regras**
- **Junção:** faturas + leitura da mesma UC e mês (`left join` por `id_uc` e `mes_referencia`). Se alguma leitura ficar sem fatura, o notebook para: o kWh dela sumiria da Gold sem aviso.
- **Nada é filtrado nem corrigido:** a base leva as flags da Silver, e a Gold que usa a métrica decide e documenta quais linhas entram (regra 5). Problema de dado vai para o dono da Silver (regra 2).
- **`ucs_atendidas`** conta todas as UCs do cadastro (hoje todas estão `ATIVA`).
- **Sem coluna de rastreio da Silver** (`_source_file`, `_ingestion_timestamp`); cada tabela ganha `_atualizado_em`.
- Antes de gravar: unicidade de cada grão, chaves sem nulos, toda UC do fato existe na `dim_uc` e reconciliação com a Silver (receita, faturas e kWh). Se algo falhar, nada é gravado.

In [0]:
%run ../bronze-to-silver/funcoes_comuns

In [0]:
dbutils.widgets.text("catalog", "workspace")
CATALOGO = dbutils.widgets.get("catalog")

In [0]:
# ---------- Regras de negócio: base ----------
P1 = ("2022-07", "2023-06")      # primeiro ano da base
P2 = ("2025-07", "2026-06")      # últimos 12 meses
CHAVE = ["id_uc", "mes_referencia"]
RASTREIO = ["_source_file", "_ingestion_timestamp"]

In [0]:
silver = lambda t: spark.table(f"{CATALOGO}.silver.tb_{t}").drop(*RASTREIO)
uc, md, gd = silver("unidades_consumidoras"), silver("medidores"), silver("geracao_distribuida")
ft, lt = silver("faturas"), silver("leituras")

In [0]:
# dim_uc: 1 linha por UC = cadastro + medidor ativo + GD
medidor = md.filter(F.col("status") == "ATIVO").select(
    "id_uc", "id_medidor", *[F.col(c).alias(f"{c}_medidor") for c in ["tipo", "modelo", "data_instalacao"]])
checar_unicidade(medidor, ["id_uc"], "medidor ativo")   # dois medidores ativos duplicariam a UC
trocas = md.groupBy("id_uc").agg((F.count("*") > 1).alias("flag_trocou_medidor"))
gd_uc = gd.select("id_uc", *[F.col(c).alias(f"gd_{c}") for c in
                             ["potencia_kwp", "modalidade", "data_conexao", "energia_injetada_kwh_media_mensal"]],
                  "flag_potencia_escala_suspeita", F.lit(True).alias("flag_tem_gd"))

dim_uc = (uc.join(medidor, "id_uc", "left").join(trocas, "id_uc", "left").join(gd_uc, "id_uc", "left")
            .fillna(False, subset=["flag_trocou_medidor", "flag_tem_gd", "flag_potencia_escala_suspeita"]))

In [0]:
# dim_alimentador: 1 linha por alimentador; ucs_atendidas é o Cc do DEC/FEC
dim_alimentador = dim_uc.groupBy("alimentador", "municipio").agg(
    F.count("*").alias("ucs_atendidas"),
    F.concat_ws(", ", F.array_sort(F.collect_set("bairro"))).alias("bairros_atendidos"))

In [0]:
# dim_mes: 1 linha por mês, gerada (não lida de uma fonte) do primeiro ao último mês faturado
ini, fim = ft.agg(F.min("mes_referencia"), F.max("mes_referencia")).first()
bandeira = ft.select("mes_referencia", "bandeira_tarifaria").distinct()
checar_unicidade(bandeira, ["mes_referencia"], "bandeira do mês")   # uma bandeira por mês

mes = F.col("mes_referencia")
dim_mes = (spark.range(1)
    .select(F.explode(F.sequence(F.to_date(F.lit(ini), "yyyy-MM"), F.to_date(F.lit(fim), "yyyy-MM"),
                                 F.expr("interval 1 month"))).alias("d"))
    .select(F.date_format("d", "yyyy-MM").alias("mes_referencia"), F.year("d").alias("ano"),
            F.month("d").alias("mes"), F.quarter("d").alias("trimestre"))
    .withColumn("periodo", F.when(mes.between(*P1), "P1").when(mes.between(*P2), "P2"))
    .join(bandeira, "mes_referencia", "left"))

In [0]:
# fato_uc_mes: 1 linha por UC × mês = fatura + leitura da mesma UC e mês
fato_uc_mes = ft.join(lt, CHAVE, "left")

In [0]:
# Checagens: falham ANTES de gravar
checar_unicidade(dim_uc, ["id_uc"], "dim_uc")
checar_unicidade(dim_alimentador, ["alimentador"], "dim_alimentador")   # alimentador em 2 municípios quebraria o Cc
checar_unicidade(dim_mes, ["mes_referencia"], "dim_mes")
checar_unicidade(fato_uc_mes, CHAVE, "fato_uc_mes")   # leitura repetida duplicaria a receita em todas as Gold
checar_sem_nulos(dim_uc, ["id_uc", "municipio", "bairro", "classe", "alimentador"], "dim_uc")
checar_sem_nulos(dim_alimentador, ["alimentador", "municipio"], "dim_alimentador")
checar_sem_nulos(fato_uc_mes, [*CHAVE, "id_fatura"], "fato_uc_mes")

sem_fatura = lt.join(ft, CHAVE, "left_anti").count()
assert sem_fatura == 0, f"{sem_fatura:,} leituras sem fatura: o kWh delas sumiria da Gold"
orfas = fato_uc_mes.join(dim_uc, "id_uc", "left_anti").count()
assert orfas == 0, f"{orfas:,} linhas do fato com UC fora da dim_uc"

# Reconciliação com a Silver (nenhuma fatura ou leitura perdida ou duplicada no caminho)
g = fato_uc_mes.agg(F.sum("valor_total"), F.count("id_fatura"), F.sum("consumo_kwh")).first()
s = (*ft.agg(F.sum("valor_total"), F.count("*")).first(), lt.agg(F.sum("consumo_kwh")).first()[0])
assert tuple(g) == s, f"reconciliação não fecha: gold={tuple(g)} silver={s}"
print(f"receita R$ {g[0]:,.2f} | faturas {g[1]:,} | kWh {g[2]:,.2f}: igual à Silver")

In [0]:
# Gravar (overwrite: reexecutável)
for tabela, df in [("dim_mes", dim_mes), ("dim_uc", dim_uc), ("dim_alimentador", dim_alimentador),
                   ("fato_uc_mes", fato_uc_mes)]:
    (df.withColumn("_atualizado_em", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOGO}.gold.{tabela}"))
    print(f"gold.{tabela}: {spark.table(f'{CATALOGO}.gold.{tabela}').count():,} linhas")

display(spark.table(f"{CATALOGO}.gold.dim_mes").orderBy("mes_referencia"))